# Fixed object-relative UVD

## Research question

This tests whether normalized horizontal position (U), vertical position (V), and boundary distance (D) provide useful object-relative structure beyond the baseline.

The visual encoder (DINOv2 ViT-S/14) and text encoder (OpenCLIP ViT-B/32) are loaded strictly from local checkpoints and remain frozen. The trainable projection, geometry-control, and decoder components are optimized from scratch for this experiment.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "final_training_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "final_model").is_dir():
    raise FileNotFoundError("Run this notebook from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
FRESH_TRAINING = True
print("Project root:", PROJECT_ROOT)
print("Training run:", RUN_ID)


Project root: /anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation
Training run: manual


## Training and stopping rule

Training uses mixed precision on CUDA, a physical batch size of 8 with two-step gradient accumulation (effective batch 16), AdamW, gradient clipping, and a validation-controlled learning-rate schedule. The maximum is 30 epochs. Training cannot stop before epoch 8 and stops after five consecutive epochs without a validation-IoU improvement greater than 0.001.

`last.pt` is saved after every epoch for interruption recovery. `best.pt` and `ui_model.pt` are selected only by `validation_seen` IoU. Test metrics never control training or checkpoint selection.

In [2]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Submit scripts/submit_full_training_fau.slurm on Alex.")
print("GPU:", torch.cuda.get_device_name(0))

from final_model.training_core import run_experiment

summary = run_experiment("fixed_uvd")
summary

GPU: NVIDIA A100-SXM4-40GB MIG 3g.20gb


/anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/dinov2/dinov2/layers/attention.py:35: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/dinov2/dinov2/layers/block.py:42: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


/anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/.venv/lib64/python3.9/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


{
  "experiment": "fixed_uvd",
  "seed": 42,
  "image_size": 224,
  "max_epochs": 30,
  "min_epochs": 8,
  "early_stopping_patience": 5,
  "early_stopping_min_delta": 0.001,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "batch_size": 8,
  "gradient_accumulation_steps": 2,
  "evaluation_batch_size": 16,
  "visual_dim": 128,
  "text_dim": 32,
  "gate_hidden_dim": 64,
  "mask_threshold": 0.5,
  "rotation_loss_weight": 0.2,
  "geometry_dropout_probability": 0.3,
  "selection_split": "validation_seen",
  "title": "Fixed object-relative UVD",
  "question": "Does explicit object-relative position and boundary distance improve the baseline?",
  "geometry": "fixed U, V, and D",
  "regularizer": "none",
  "run_id": "manual",
  "device": "cuda:0",
  "gpu": "NVIDIA A100-SXM4-40GB MIG 3g.20gb",
  "gpu_count": 1,
  "amp": "float16",
  "cudnn_benchmark": true,
  "python": "3.9.25",
  "torch": "2.8.0+cu128",
  "dino_checkpoint": "models/pretrained/dinov2_vits14_pretrain.pth",
  "clip_checkpoin

train:   0%|          | 0/4088 [00:00<?, ?it/s]

...

[fixed_uvd] 17/30 train IoU=0.3752 val IoU=0.2855 val Dice=0.3789 patience=5/5 time=5.4m peak=0.7GB
Early stopping at epoch 17; best epoch was 12.


test_seen:   0%|          | 0/211 [00:00<?, ?it/s]

test_unseen:   0%|          | 0/100 [00:00<?, ?it/s]

experiment       split  samples      iou     dice  leakage  selected_epoch  validation_iou  validation_dice
 fixed_uvd   test_seen     3371 0.298380 0.398249 0.197430              12        0.285524         0.380421
 fixed_uvd test_unseen     1586 0.257012 0.350158 0.159851              12        0.285524         0.380421
Best checkpoint: /anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/training_results/fixed_uvd/best.pt
UI checkpoint: /anvme/workspace/v123be52-cv_project/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/training_results/fixed_uvd/ui_model.pt


,experiment,split,samples,iou,dice,leakage,selected_epoch,validation_iou,validation_dice
0,fixed_uvd,test_seen,3371,0.298380,0.398249,0.197430,12,0.285524,0.380421
1,fixed_uvd,test_unseen,1586,0.257012,0.350158,0.159851,12,0.285524,0.380421


## Produced evidence

This notebook writes its checkpoint to `training_results/fixed_uvd/` and its metrics, per-example predictions, configuration, training curves, and unseen qualitative examples to `training_results/fixed_uvd/`. Re-execution with the same run ID resumes from the last completed epoch.

## Saved figures

![Training curves](../fixed_uvd/training_curves.png)

![Seen and unseen metrics](../fixed_uvd/evaluation_comparison.png)

![Qualitative unseen predictions](../fixed_uvd/qualitative_unseen.png)
